In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs

In [2]:
dirs='~/shared/ugit0034/Zhankun/202608/'
diro='~/scratch/ML4O2_temp/'
pfm=['OSD','CTD','PFL']
yr0=[1965,1971,2002]
yr1=[2025,2025,2025]
z1=[137,117,105]
#print(np.round(1.49))
ds=xr.open_dataset(dirs+'Oxygen_OSD_2005.nc')
z=ds['depth'].to_numpy()
np.save('depth.npy',z)
#ds

In [3]:
def bin_data(fn,year,stype):
    # stype: 0=OSD, 1=CTD, 2=PFL
    z=np.load('depth.npy')
    ds=xr.open_dataset(dirs+fn)
    Np=np.size(ds.profile)
    Nz=np.size(ds.depth)
    print(fn,Np,Nz)
    tmptrc=np.zeros((12,137,180,360))
    tmplon=np.zeros((12,180,360))
    tmplat=np.zeros((12,180,360))
    tmpcnt=np.zeros((12,137,180,360),dtype=np.int32)
    tmpcntxy=np.zeros((12,180,360),dtype=np.int32)
    for n in range(Np):
        trc=np.zeros(137)
        trc0=ds.Oxygen[n,:].to_numpy()
        spass=True
        if m==2:
            smode=ds.mode[n].to_numpy()
            if smode==2: # delayed-mode
                spass=True
            else:
                spass=False
        Nz0=np.size(trc0)
        trc[:Nz0]=trc0
        lon=ds.lon[n].to_numpy()
        lat=ds.lat[n].to_numpy()
        mon=ds.month[n].to_numpy()-1
        i0=int(np.round(lon)+180)
        # ad-hoc fix for longitude error
        if i0==360:
            i0=359
        j0=int(np.round(lat)+90)
        if (i0>=0)&(i0<360)&(j0>=0)&(j0<180)&(spass==True): 
            cntz=np.where(np.isnan(trc)==False,1,0)
            trcz=np.where(np.isnan(trc)==False,trc,0)
            tmptrc[mon,:,j0,i0]=tmptrc[mon,:,j0,i0]+trcz
            tmpcnt[mon,:,j0,i0]=tmpcnt[mon,:,j0,i0]+cntz
            tmpcntxy[mon,j0,i0]=tmpcntxy[mon,j0,i0]+1
            tmplat[mon,j0,i0]=tmplat[mon,j0,i0]+lat
            tmplon[mon,j0,i0]=tmplon[mon,j0,i0]+lon
        #else:
        #    print(f'--warning--({i0},{j0}) or non-DM')
    #
    trcbin = tmptrc/tmpcnt
    lonbin = tmplon/tmpcntxy
    latbin = tmplat/tmpcntxy
    #
    time=np.arange(f'{year}-01',f'{year+1}-01',dtype='datetime64[M]')
    x=np.arange(-180,180,1)+0.5
    y=np.arange(-90,90,1)+0.5
    da=xr.DataArray(data=trcbin,name='Oxygen',dims=['time','depth','lat','lon'],
                   coords={'time':time,'depth':z,'lat':y,'lon':x})
    ds=da.to_dataset()
    ds['sample_count']=xr.DataArray(data=tmpcnt,dims=['time','depth','lat','lon'],
                   coords={'time':time,'depth':z,'lat':y,'lon':x})
    ds['profile_count']=xr.DataArray(data=tmpcntxy,dims=['time','lat','lon'],
                   coords={'time':time,'lat':y,'lon':x})
    ds['latitude']=xr.DataArray(data=latbin,dims=['time','lat','lon'],
                   coords={'time':time,'lat':y,'lon':x})
    ds['longitude']=xr.DataArray(data=lonbin,dims=['time','lat','lon'],
                   coords={'time':time,'lat':y,'lon':x})
    ds.to_netcdf(f'{diro}binned_'+fn)
    return 1

In [6]:
for m in [0,1,2]:
    yrs=np.arange(yr0[m],yr1[m]+1,1)
    #
    for n,year in enumerate(yrs):
        fn=f'Oxygen_{pfm[m]}_{year}.nc'
        dummy=bin_data(fn,year,m)

Oxygen_OSD_1965.nc 17446 137


/tmp/ipykernel_1290874/2014544849.py:44: RuntimeWarning: invalid value encountered in divide
  trcbin = tmptrc/tmpcnt
/tmp/ipykernel_1290874/2014544849.py:45: RuntimeWarning: invalid value encountered in divide
  lonbin = tmplon/tmpcntxy
/tmp/ipykernel_1290874/2014544849.py:46: RuntimeWarning: invalid value encountered in divide
  latbin = tmplat/tmpcntxy


Oxygen_OSD_1966.nc 17698 137


/tmp/ipykernel_1290874/2014544849.py:44: RuntimeWarning: invalid value encountered in divide
  trcbin = tmptrc/tmpcnt
/tmp/ipykernel_1290874/2014544849.py:45: RuntimeWarning: invalid value encountered in divide
  lonbin = tmplon/tmpcntxy
/tmp/ipykernel_1290874/2014544849.py:46: RuntimeWarning: invalid value encountered in divide
  latbin = tmplat/tmpcntxy


Oxygen_OSD_1967.nc 17738 137


...

/tmp/ipykernel_1290874/2014544849.py:44: RuntimeWarning: invalid value encountered in divide
  trcbin = tmptrc/tmpcnt
/tmp/ipykernel_1290874/2014544849.py:45: RuntimeWarning: invalid value encountered in divide
  lonbin = tmplon/tmpcntxy
/tmp/ipykernel_1290874/2014544849.py:46: RuntimeWarning: invalid value encountered in divide
  latbin = tmplat/tmpcntxy


Oxygen_CTD_2024.nc 1299 119


/tmp/ipykernel_1290874/2014544849.py:44: RuntimeWarning: invalid value encountered in divide
  trcbin = tmptrc/tmpcnt
/tmp/ipykernel_1290874/2014544849.py:45: RuntimeWarning: invalid value encountered in divide
  lonbin = tmplon/tmpcntxy
/tmp/ipykernel_1290874/2014544849.py:46: RuntimeWarning: invalid value encountered in divide
  latbin = tmplat/tmpcntxy


Oxygen_CTD_2025.nc 215 119


/tmp/ipykernel_1290874/2014544849.py:44: RuntimeWarning: invalid value encountered in divide
  trcbin = tmptrc/tmpcnt
/tmp/ipykernel_1290874/2014544849.py:45: RuntimeWarning: invalid value encountered in divide
  lonbin = tmplon/tmpcntxy
/tmp/ipykernel_1290874/2014544849.py:46: RuntimeWarning: invalid value encountered in divide
  latbin = tmplat/tmpcntxy


In [ ]:
ds1=xr.open_mfdataset(f'{diro}binned_Oxygen_PFL_*.nc')
ds1.to_netcdf('Oxygen_PFL_1x1bin_2002-2025.nc')

In [ ]:
ds1=xr.open_mfdataset(f'{diro}binned_Oxygen_OSD_*.nc')
ds1.to_netcdf('Oxygen_OSD_1x1bin_1965-2025.nc')

In [ ]:
ds1=xr.open_mfdataset(f'{diro}binned_Oxygen_CTD_*.nc')
ds1.to_netcdf('Oxygen_CTD_1x1bin_1971-2025.nc')

In [ ]:
########